In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "LTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-09-01 00:00:00+00:00,108.87,108.89,108.81,108.89,110.284,2025-09-01 00:00:59.999999+00:00,12004.09154,159,78.063,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000000,0.000000,0.000000,NaN,NaN
1,2025-09-01 00:01:00+00:00,108.90,108.98,108.90,108.98,143.091,2025-09-01 00:01:59.999999+00:00,15584.65490,141,131.541,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.002019,0.001122,0.000897,NaN,NaN
2,2025-09-01 00:02:00+00:00,108.98,108.98,108.86,108.91,60.387,2025-09-01 00:02:59.999999+00:00,6577.87439,143,7.238,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000402,0.000827,-0.000425,NaN,NaN
3,2025-09-01 00:03:00+00:00,108.91,108.93,108.87,108.88,356.131,2025-09-01 00:03:59.999999+00:00,38780.88348,170,263.167,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.001426,0.000064,-0.001490,NaN,NaN
4,2025-09-01 00:04:00+00:00,108.88,108.88,108.66,108.67,382.463,2025-09-01 00:04:59.999999+00:00,41584.76818,246,118.148,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.010746,-0.003152,-0.007594,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 284,601
[info] optuna train rows: 182,144
[info] valid rows:        45,536
[info] test rows:         56,921


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-20 05:36:07,047] A new study created in memory with name: no-name-73ed104d-9010-4087-9223-22b879e3b27d


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:08<?, ?it/s]

Best trial: 0. Best value: -0.012322:   0%|          | 0/50 [00:08<?, ?it/s]

Best trial: 0. Best value: -0.012322:   2%|▏         | 1/50 [00:08<06:33,  8.03s/it]

...

Best trial: 41. Best value: 0.0338893:  98%|█████████▊| 49/50 [31:32<00:36, 36.10s/it]

Best trial: 41. Best value: 0.0338893:  98%|█████████▊| 49/50 [31:32<00:36, 36.10s/it]

Best trial: 41. Best value: 0.0338893: 100%|██████████| 50/50 [31:32<00:00, 30.89s/it]

Best trial: 41. Best value: 0.0338893: 100%|██████████| 50/50 [31:32<00:00, 37.85s/it]

[I 2026-03-20 06:07:39,630] Trial 49 finished with value: 0.02491180713758575 and parameters: {'n_estimators': 300, 'max_depth': 18, 'min_samples_split': 18, 'min_samples_leaf': 3, 'max_features': 0.3, 'bootstrap': False}. Best is trial 41 with value: 0.03388932416343065.

[optuna] best trial
value: 0.033889
params:
  n_estimators: 700
  max_depth: 17
  min_samples_split: 9
  min_samples_leaf: 14
  max_features: 0.8
  bootstrap: True


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final rf...


[training] done in 59.79s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:      0.426684
Test IC:       0.000447
Train Rank IC: 0.093407
Test Rank IC:  0.032142
Train RMSE:    0.002985
Test RMSE:     0.001991


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
mom_60              0.273654
mom_x_imb           0.100474
mom_30              0.092848
dist_ma_5           0.073084
mom_3               0.063906
atr_norm            0.058062
range_15            0.032011
vol_30              0.029331
vol_15              0.026824
mom_15              0.024123
mom_5               0.022018
dist_ma_30          0.019041
macd_hist           0.017847
mom_10              0.015966
range_5             0.015558
imbalance_15        0.014625
dom_sin             0.013052
vol_5               0.012732
dist_ma_15          0.011484
bar_range           0.011280
vol_regime_ratio    0.006894
hour_cos            0.005614
imbalance_5         0.005567
trend_strength      0.004992
dom_cos             0.004804
vol_ratio_5_30      0.004408
trend_x_imb         0.004287
range_ratio         0.003931
dist_ma_15_z        0.003799
mr_x_vol            0.003356
trades_z            0.003320
hour_sin            0.003210
dow_sin             0.003200
month_sin  

In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/rf/LTCUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/rf/LTCUSDT__h5_model.joblib
[saved] features -> models/rf/LTCUSDT__h5_feature_cols.json
[saved] feature importance -> models/rf/LTCUSDT__h5_feature_importance.csv
[saved] metadata -> models/rf/LTCUSDT__h5_meta.json
